# Раздел 5 · Фильтрация по payload и payload-индексы

In [ ]:
%pip install qdrant-client sentence-transformers ipykernel --quiet


In [1]:
import json
import subprocess
import numpy as np

from sentence_transformers import SentenceTransformer

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue, MatchAny, Range, PayloadSchemaType
)

print("OK: библиотеки импортированы")


/home/dimon/vscode/examples_qdrant/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OK: библиотеки импортированы


### Docker: тот же сервер, что и в остальных ноутбуках курса

In [2]:
CONTAINER_NAME = "qdrant_lecture"


def sh(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return result.stdout.strip(), result.stderr.strip(), result.returncode


sh("docker volume create qdrant_storage")

_, _, exists = sh("docker inspect {name}".format(name=CONTAINER_NAME))
if exists != 0:
    out, err, _ = sh(
        "docker run -d --name {name} -p 6333:6333 -p 6334:6334 "
        "-v qdrant_storage:/qdrant/storage qdrant/qdrant".format(name=CONTAINER_NAME)
    )
else:
    out, err, _ = sh("docker start {name}".format(name=CONTAINER_NAME))
print(out or err)

client = QdrantClient(url="http://localhost:6333")
print(client.get_collections())


qdrant_lecture
collections=[CollectionDescription(name='exercise1_books'), CollectionDescription(name='products'), CollectionDescription(name='products_20260928'), CollectionDescription(name='products_20261001_bm25_fix_ru'), CollectionDescription(name='products_20261002'), CollectionDescription(name='products_base'), CollectionDescription(name='products_cm'), CollectionDescription(name='terraforming_plans')]


In [4]:
model = SentenceTransformer("intfloat/multilingual-e5-small")
EMBED_DIM = model.get_embedding_dimension()


def embed_passages(texts):
    return model.encode(["passage: " + t for t in texts], normalize_embeddings=True).tolist()


def embed_query(text):
    return model.encode("query: " + text, normalize_embeddings=True).tolist()


def cos(x, y):
    return float(np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y)))


with open("books_dataset.json", encoding="utf-8") as f:
    BOOKS = json.load(f)
BOOK_BY_ID = {b["id"]: b for b in BOOKS}
PREFIX = "exercise1_"
BOOKS_COLLECTION = PREFIX + "books"

print("Размерность эмбеддинга:", EMBED_DIM, "| книг в подборке:", len(BOOKS))


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20558.79it/s]


Размерность эмбеддинга: 384 | книг в подборке: 87


In [5]:
assert client.collection_exists(BOOKS_COLLECTION), (
    "Коллекция {} ещё не создана. Откройте и выполните ноутбук "
    "Раздела 3 (\"Коллекции и точки\") первым - там она создаётся и наполняется."
).format(BOOKS_COLLECTION)
print("OK:", BOOKS_COLLECTION, "-", client.count(BOOKS_COLLECTION).count, "точек")


OK: exercise1_books - 87 точек


## Фильтрация по payload

### Пример — must / must_not

`genre = "повесть"`, `year >= 1830`, `year <= 1870`, ранжируем запросом **"история о любви и чести"**:

In [6]:
# query_filter применяется ДО ранжирования по вектору - сначала отсекаем по жанру и году, потом сортируем по смыслу
example_filtered = client.query_points(
    collection_name=BOOKS_COLLECTION,
    query=embed_query("история о любви и чести"),
    query_filter=Filter(
        must=[  # must = И: обязаны выполняться оба условия
            FieldCondition(key="genre", match=MatchValue(value="повесть")),
            FieldCondition(key="year", range=Range(gte=1830, lte=1870)),
        ]
    ),
    limit=10,
).points

print([(h.id, h.payload["title"], h.payload["year"]) for h in example_filtered])


[(30, 'Первая любовь', 1860), (29, 'Ася', 1858), (24, 'Тарас Бульба', 1835), (26, 'Капитанская дочка', 1836), (23, 'Записки сумасшедшего', 1835), (28, 'Дубровский', 1833), (40, 'Леди Макбет Мценского уезда', 1865), (25, 'Вий', 1835), (22, 'Нос', 1836), (27, 'Пиковая дама', 1834)]


Реальный результат: 10 произведений, `id` `{30, 29, 24, 26, 23, 28, 40, 25, 22, 27}` — включая
"Асю" Тургенева, "Тараса Бульбу" Гоголя, "Дубровского" Пушкина.

## Задание 1

Тот же приём для `genre = "рассказ"`, `year >= 1880`, `year <= 1900`, запрос **"человек, застрявший в рутине и мелочности"**. Сохраните в `hits_ex1`.

**Используйте:** `client.query_points(...)`, `Filter(must=[...])`, `FieldCondition(...)`, `MatchValue(...)`, `Range(...)`.

In [9]:
# ВАШ КОД ЗДЕСЬ
hits_ex1 = client.query_points(
    collection_name=BOOKS_COLLECTION,
    query=embed_query("человек, застрявший в рутине и мелочности"),
    query_filter=Filter(
        must=[  # must = И: обязаны выполняться оба условия
            FieldCondition(key="genre", match=MatchValue(value="рассказ")),
            FieldCondition(key="year", range=Range(gte=1880, lte=1900)),
        ]
    ),
    limit=10,
).points

print([(h.id, h.payload["title"], h.payload["year"]) for h in example_filtered])


[(30, 'Первая любовь', 1860), (29, 'Ася', 1858), (24, 'Тарас Бульба', 1835), (26, 'Капитанская дочка', 1836), (23, 'Записки сумасшедшего', 1835), (28, 'Дубровский', 1833), (40, 'Леди Макбет Мценского уезда', 1865), (25, 'Вий', 1835), (22, 'Нос', 1836), (27, 'Пиковая дама', 1834)]


In [10]:
def check_ex1(hits_ex1):
    for h in hits_ex1:
        assert h.payload["genre"] == "рассказ"
        assert 1880 <= h.payload["year"] <= 1900
    print("OK:", [(h.id, h.payload["title"], h.payload["year"]) for h in hits_ex1])
    print("ОТВЕТ ДЛЯ STEPIK:", hits_ex1[0].id)


check_ex1(hits_ex1)


OK: [(46, 'Ионыч', 1898), (49, 'Унтер Пришибеев', 1885), (44, 'Человек в футляре', 1898), (52, 'Спать хочется', 1888), (48, 'Смерть чиновника', 1883), (57, 'Чудесный доктор', 1897), (50, 'Злоумышленник', 1885), (43, 'Толстый и тонкий', 1883), (47, 'Крыжовник', 1898), (59, 'Челкаш', 1895)]
ОТВЕТ ДЛЯ STEPIK: 46


### Пример — payload-индексы и facet

Индексы нужны один раз на коллекцию — создаём их здесь, дальше они доступны во всех упражнениях этой части:

In [11]:
# индекс на поле - не обязателен для фильтрации как таковой, но сильно ускоряет её на больших коллекциях
# и обязателен для facet(); один раз на коллекцию, а не на каждый запрос
client.create_payload_index(BOOKS_COLLECTION, field_name="genre", field_schema=PayloadSchemaType.KEYWORD)
client.create_payload_index(BOOKS_COLLECTION, field_name="author", field_schema=PayloadSchemaType.KEYWORD)
client.create_payload_index(BOOKS_COLLECTION, field_name="year", field_schema=PayloadSchemaType.INTEGER)
client.create_payload_index(BOOKS_COLLECTION, field_name="pages", field_schema=PayloadSchemaType.INTEGER)
client.create_payload_index(BOOKS_COLLECTION, field_name="in_school_curriculum", field_schema=PayloadSchemaType.BOOL)

# facet - аналог "SELECT genre, COUNT(*) GROUP BY genre" на уровне payload-индекса, без вектора поиска
example_facet = client.facet(collection_name=BOOKS_COLLECTION, key="genre")
print({v.value: v.count for v in example_facet.hits})


{'повесть': 21, 'роман': 20, 'рассказ': 18, 'пьеса': 15, 'поэма': 13}


У `genre` всего 5 значений, поэтому пример сработал "как есть". Но у `client.facet` есть параметр
`limit` со значением по умолчанию **10** — если различных значений в поле больше, часть просто не
попадёт в ответ, без предупреждения.

## Задание 2

Получите facet по полю `author`. В датасете 18 разных авторов — не забудьте увеличить `limit`, иначе часть авторов потеряется молча. Сохраните в `facet_ex2`.

**Используйте:** `client.facet(...)`.

In [14]:
# ВАШ КОД ЗДЕСЬ
facet_ex2 = client.facet(collection_name=BOOKS_COLLECTION, key="author", limit=18)
print({v.value: v.count for v in facet_ex2.hits})


{'Антон Чехов': 19, 'Александр Пушкин': 9, 'Иван Тургенев': 9, 'Лев Толстой': 9, 'Николай Гоголь': 8, 'Фёдор Достоевский': 5, 'Александр Островский': 4, 'Михаил Лермонтов': 4, 'Александр Куприн': 3, 'Иван Гончаров': 3, 'Максим Горький': 3, 'Николай Некрасов': 3, 'Василий Жуковский': 2, 'Николай Лесков': 2, 'Александр Грибоедов': 1, 'Александр Сухово-Кобылин': 1, 'Михаил Салтыков-Щедрин': 1, 'Николай Чернышевский': 1}


In [ ]:
def check_ex2(facet_ex2):
    counts = {v.value: v.count for v in facet_ex2.hits}
    assert len(counts) == 18, "в датасете 18 авторов, получили {} — проверьте параметр limit".format(len(counts))
    assert sum(counts.values()) == 87, "сумма facet-счётчиков должна быть 87, получили {}".format(sum(counts.values()))
    assert counts.get("Антон Чехов") == 19, "ожидали 19 произведений Чехова, получили {}".format(counts.get("Антон Чехов"))
    print("OK:", counts)
    print("ОТВЕТ ДЛЯ STEPIK:", counts.get("Антон Чехов"))


check_ex2(facet_ex2)


OK: {'Антон Чехов': 19, 'Александр Пушкин': 9, 'Иван Тургенев': 9, 'Лев Толстой': 9, 'Николай Гоголь': 8, 'Фёдор Достоевский': 5, 'Александр Островский': 4, 'Михаил Лермонтов': 4, 'Александр Куприн': 3, 'Иван Гончаров': 3, 'Максим Горький': 3, 'Николай Некрасов': 3, 'Василий Жуковский': 2, 'Николай Лесков': 2, 'Александр Грибоедов': 1, 'Александр Сухово-Кобылин': 1, 'Михаил Салтыков-Щедрин': 1, 'Николай Чернышевский': 1}
ОТВЕТ ДЛЯ STEPIK: 19


### Пример — Range + MatchAny, чистый count

Без вектора поиска — просто сколько произведений подходит под фильтр: `year` в `[1880, 1910]` и `genre` из `["роман", "повесть"]`:

In [16]:
# client.count с фильтром, но без query - чистый подсчёт, векторный поиск тут вообще не участвует
example_count = client.count(
    collection_name=BOOKS_COLLECTION,
    count_filter=Filter(
        must=[
            FieldCondition(key="year", range=Range(gte=1880, lte=1910)),
            FieldCondition(key="genre", match=MatchAny(any=["роман", "повесть"])),  # MatchAny = "любое из этих значений" (OR внутри одного условия)
        ]
    ),
).count
print("count:", example_count)


count: 9


Реальный результат: `9`.

## Задание 3

Посчитайте число произведений с `year` в диапазоне `[1830, 1860]` **и** `genre` из списка `["поэма", "пьеса"]`. Сохраните в `count_ex3`.

**Используйте:** `client.count(...)`, `Filter(must=[...])`, `Range(...)`, `MatchAny(...)`.

In [17]:
# ВАШ КОД ЗДЕСЬ
count_ex3 = client.count(
    collection_name=BOOKS_COLLECTION,
    count_filter=Filter(
        must=[
            FieldCondition(key="year", range=Range(gte=1830, lte=1860)),
            FieldCondition(key="genre", match=MatchAny(any=["поэма", "пьеса"])),  # MatchAny = "любое из этих значений" (OR внутри одного условия)
        ]
    ),
).count
print("count:", count_ex3)



count: 8


In [18]:
def check_ex3(count_ex3):
    assert count_ex3 == 8, "ожидали 8, получили {}".format(count_ex3)
    print("OK: count =", count_ex3)
    print("ОТВЕТ ДЛЯ STEPIK:", count_ex3)


check_ex3(count_ex3)


OK: count = 8
ОТВЕТ ДЛЯ STEPIK: 8
